# 03 — XAUUSD H1 Hidden Markov Model (HMM) Regime Detection

## Research objective

This notebook tests whether a **three-state Gaussian Hidden Markov Model (HMM)** can identify latent Bullish, Bearish, and Sideways market regimes in XAUUSD H1 data. It then evaluates the existing N-Method setups under those HMM regimes and compares an HMM-based strategy selection with the existing EMA/ADX baseline.

This is a **chronological holdout experiment**. The HMM is fitted only on the first 60% of usable feature observations. Its parameters and state-to-regime mapping are then frozen. The remaining 40% is classified one observation at a time using a forward filter. No future observations from the holdout segment are used to classify an earlier holdout candle.

> **Research status:** historical exploratory analysis only. This notebook does not establish a validated trading system or evidence of live profitability.

## Method design

### HMM observations

The model uses four features calculated from completed H1 candles:

1. `log_return_1h`: one-bar log return.
2. `momentum_24h`: 24-bar cumulative log return.
3. `realized_volatility_24h`: rolling standard deviation of one-bar log returns over 24 bars.
4. `efficiency_24h`: absolute 24-bar price change divided by the sum of absolute one-bar price changes over the same period.

The inputs are standardized using training-period statistics only. A three-state `GaussianHMM` with diagonal covariance is used as a deliberately simple baseline.

### Assigning economic labels to latent states

HMM state IDs are arbitrary and can change between model fits. After training, states are ranked by their posterior-weighted mean `momentum_24h` calculated on the training sample only:

- Lowest mean momentum → **Bearish**
- Middle mean momentum → **Sideways**
- Highest mean momentum → **Bullish**

This is a transparent naming convention, not a ground-truth classification. The state profile table must be inspected: a state labelled Sideways may still have elevated volatility or directional efficiency.

### Avoiding look-ahead in the holdout period

`GaussianHMM.predict_proba()` performs forward-backward inference over the supplied sequence, so it can use later observations in that sequence. It is used below only on the historical training sample to calculate training diagnostics and initialize the filter at the end of training. Holdout probabilities are calculated with a **forward-only recursive filter**, one H1 observation at a time.

The HMM is not re-fitted during the holdout period. This is a fixed-parameter chronological holdout, not a rolling-refit walk-forward test. Re-estimation policy can be studied separately after this baseline is understood.

References: [hmmlearn GaussianHMM API](https://hmmlearn.readthedocs.io/en/stable/api.html) and [hmmlearn tutorial](https://hmmlearn.readthedocs.io/en/stable/tutorial.html).

In [3]:
from __future__ import annotations

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.special import logsumexp

try:
    from hmmlearn.hmm import GaussianHMM
except ImportError as exc:
    raise ImportError(
        "Missing dependency 'hmmlearn'. From the project root, run: "
        "python -m pip install -r requirements.txt"
    ) from exc

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "XAUUSD_H1.csv"
TRADES_PATH = PROJECT_ROOT / "outputs" / "XAUUSD_H1_N_method_trades.csv"
BASELINE_TRADES_PATH = PROJECT_ROOT / "outputs" / "XAUUSD_H1_N_method_trades_with_regime.csv"
BASELINE_CANDLES_PATH = PROJECT_ROOT / "outputs" / "XAUUSD_H1_candles_with_regime.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
ASSET_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ASSET_DIR.mkdir(parents=True, exist_ok=True)

for path in [DATA_PATH, TRADES_PATH, BASELINE_TRADES_PATH, BASELINE_CANDLES_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. Run notebooks 01 and 02 first, "
            "and confirm the source CSV is in the project's data/ folder."
        )

print("Project root:", PROJECT_ROOT)
print("GaussianHMM imported successfully.")

ModuleNotFoundError: No module named 'scipy'

In [ ]:
# Reproducible baseline settings. Keep these fixed for this experiment.
N_STATES = 3
TRAIN_FRACTION = 0.60
HMM_MAX_ITER = 100
HMM_TOL = 1e-3
HMM_RESTART_SEEDS = [7, 21, 42]
COVARIANCE_TYPE = "diag"

FEATURE_LOOKBACK = 24  # bars, not a guaranteed 24 clock hours across market gaps
VALID_RESULTS = {"TP", "SL", "OPEN_AT_END"}
REGIME_ORDER = ["Bullish", "Bearish", "Sideways"]
REGIME_COLORS = {
    "Bullish": "tab:green",
    "Bearish": "tab:red",
    "Sideways": "tab:gray",
}

print({
    "n_states": N_STATES,
    "training_fraction": TRAIN_FRACTION,
    "max_iterations": HMM_MAX_ITER,
    "restart_seeds": HMM_RESTART_SEEDS,
    "covariance_type": COVARIANCE_TYPE,
})

## 1. Load H1 candles and previously generated N-Method trades

The source file is headerless and is interpreted as `Date, Open, High, Low, Close, Volume`. The trade-level CSVs are generated by notebooks 01 and 02; no new N-Method signals are created in this notebook.

In [ ]:
raw = pd.read_csv(DATA_PATH, header=None)
if raw.shape[1] < 5:
    raise ValueError("The H1 source CSV must have at least five columns: Date, Open, High, Low, Close.")

columns = ["Date", "Open", "High", "Low", "Close"]
if raw.shape[1] >= 6:
    columns.append("Volume")
raw = raw.iloc[:, :len(columns)].copy()
raw.columns = columns
raw["Date"] = pd.to_datetime(raw["Date"], errors="coerce")
for col in ["Open", "High", "Low", "Close"]:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")
if "Volume" in raw.columns:
    raw["Volume"] = pd.to_numeric(raw["Volume"], errors="coerce")

data = (
    raw.dropna(subset=["Date", "Open", "High", "Low", "Close"])
       .sort_values("Date")
       .drop_duplicates("Date")
       .set_index("Date")
)

invalid = (data["High"] < data[["Open", "Close", "Low"]].max(axis=1)) | (
    data["Low"] > data[["Open", "Close", "High"]].min(axis=1)
)
if invalid.any():
    raise ValueError(f"Found {invalid.sum()} invalid OHLC rows.")

trades = pd.read_csv(TRADES_PATH, parse_dates=["signal_date", "entry_date", "exit_date"])
trades_baseline = pd.read_csv(
    BASELINE_TRADES_PATH,
    parse_dates=["signal_date", "entry_date", "exit_date"]
)
if "Regime" not in trades_baseline.columns:
    raise ValueError("The baseline trade file must contain the 'Regime' column. Run notebook 02 first.")

baseline_candles = pd.read_csv(BASELINE_CANDLES_PATH, parse_dates=["Date"]).set_index("Date")
if "Regime" not in baseline_candles.columns:
    raise ValueError("The baseline candle file must contain the 'Regime' column.")

print(f"H1 bars: {len(data):,}")
print(f"Period: {data.index.min()} to {data.index.max()}")
print(f"Raw N-Method trade records: {len(trades):,}")
display(data.head())

## 2. Construct causal HMM features

Each feature at timestamp *t* uses the close at *t* and historical observations only. Rolling windows are backward-looking. The timestamp convention in the supplied source is not documented, so the model's regime is attached to the same index timestamp used by the backtest's `signal_date`.

In [ ]:
close = data["Close"].astype(float)
log_return = np.log(close / close.shift(1))

feature_frame = pd.DataFrame(index=data.index)
feature_frame["log_return_1h"] = log_return
feature_frame["momentum_24h"] = np.log(close / close.shift(FEATURE_LOOKBACK))
feature_frame["realized_volatility_24h"] = log_return.rolling(
    FEATURE_LOOKBACK, min_periods=FEATURE_LOOKBACK
).std()

absolute_path = close.diff().abs().rolling(
    FEATURE_LOOKBACK, min_periods=FEATURE_LOOKBACK
).sum()
net_move = (close - close.shift(FEATURE_LOOKBACK)).abs()
feature_frame["efficiency_24h"] = net_move / absolute_path.replace(0, np.nan)

feature_frame = feature_frame.replace([np.inf, -np.inf], np.nan).dropna()
FEATURE_COLUMNS = [
    "log_return_1h",
    "momentum_24h",
    "realized_volatility_24h",
    "efficiency_24h",
]

if len(feature_frame) < 5_000:
    raise ValueError("Too few usable feature observations for a meaningful three-state HMM experiment.")

print(f"Usable feature observations: {len(feature_frame):,}")
print(f"Feature period: {feature_frame.index.min()} to {feature_frame.index.max()}")
display(feature_frame[FEATURE_COLUMNS].describe().T)

## 3. Chronological train / holdout split and training-only scaling

The first 60% of usable feature observations are the training sample. The remaining 40% is reserved as a chronological holdout. The scaler's mean and standard deviation are estimated on training observations only and then applied unchanged to the holdout.

In [ ]:
split_position = int(len(feature_frame) * TRAIN_FRACTION)
if split_position <= 1 or split_position >= len(feature_frame) - 1:
    raise ValueError("Invalid chronological train/holdout split.")

train_features = feature_frame.iloc[:split_position].copy()
holdout_features = feature_frame.iloc[split_position:].copy()

train_raw = train_features[FEATURE_COLUMNS].to_numpy(dtype=float)
holdout_raw = holdout_features[FEATURE_COLUMNS].to_numpy(dtype=float)

scaler_mean = train_raw.mean(axis=0)
scaler_scale = train_raw.std(axis=0, ddof=0)
scaler_scale = np.where(scaler_scale < 1e-12, 1.0, scaler_scale)

X_train = (train_raw - scaler_mean) / scaler_scale
X_holdout = (holdout_raw - scaler_mean) / scaler_scale

train_end = train_features.index[-1]
holdout_start = holdout_features.index[0]
print(f"Training observations: {len(train_features):,}")
print(f"Training period: {train_features.index.min()} to {train_end}")
print(f"Holdout observations: {len(holdout_features):,}")
print(f"Holdout period: {holdout_start} to {holdout_features.index.max()}")

## 4. Fit a three-state Gaussian HMM

Several fixed random seeds are tried because HMM fitting can converge to different local optima. The model with the highest training log-likelihood is retained. All candidate fits use the same number of states, feature set, covariance structure, and iteration settings.

The training log-likelihood is used only to choose among these initializations; it is not an out-of-sample performance metric.

In [ ]:
fit_results = []
best_model = None
best_score = -np.inf

for seed in HMM_RESTART_SEEDS:
    candidate = GaussianHMM(
        n_components=N_STATES,
        covariance_type=COVARIANCE_TYPE,
        n_iter=HMM_MAX_ITER,
        tol=HMM_TOL,
        min_covar=1e-3,
        random_state=seed,
        verbose=False,
    )
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            candidate.fit(X_train)
        score = float(candidate.score(X_train))
        fit_results.append({
            "seed": seed,
            "train_log_likelihood": score,
            "iterations": int(candidate.monitor_.iter),
            "converged": bool(candidate.monitor_.converged),
            "warnings": " | ".join(str(w.message) for w in caught),
        })
        if np.isfinite(score) and score > best_score:
            best_score = score
            best_model = candidate
    except Exception as exc:
        fit_results.append({
            "seed": seed,
            "train_log_likelihood": np.nan,
            "iterations": np.nan,
            "converged": False,
            "warnings": f"FIT FAILED: {type(exc).__name__}: {exc}",
        })

fit_diagnostics = pd.DataFrame(fit_results)
if best_model is None:
    raise RuntimeError("All HMM initializations failed. Inspect fit_diagnostics and consider changing the initialization settings.")

print(f"Selected random seed: {best_model.random_state}")
print(f"Best training log-likelihood: {best_score:,.2f}")
display(fit_diagnostics)

## 5. Map latent states to Bullish, Bearish, and Sideways

The HMM's numeric state IDs have no inherent economic meaning. This section calculates posterior-weighted feature profiles on the **training sample only**, then assigns descriptive regime names using the training-period mean 24-bar momentum. No holdout outcomes are used for state mapping.

In [ ]:
# Training posterior probabilities may use forward-backward information within the
# training sample. This is acceptable here because state mapping is frozen before
# the holdout begins. Holdout classification below uses a separate forward filter.
train_smoothed_prob = best_model.predict_proba(X_train)

profile_rows = []
for state_id in range(N_STATES):
    weights = train_smoothed_prob[:, state_id]
    weight_sum = weights.sum()
    if weight_sum <= 0:
        raise RuntimeError(f"State {state_id} has zero posterior mass in the training sample.")
    weighted_means = {
        col: float(np.average(train_features[col].to_numpy(), weights=weights))
        for col in FEATURE_COLUMNS
    }
    profile_rows.append({
        "State": state_id,
        "Posterior occupancy %": 100 * float(weights.mean()),
        **weighted_means,
    })

state_profiles = pd.DataFrame(profile_rows).set_index("State")
# Rank by the posterior-weighted average 24-bar momentum.
ranked_states = state_profiles["momentum_24h"].sort_values(kind="stable").index.tolist()
state_to_regime = {
    int(ranked_states[0]): "Bearish",
    int(ranked_states[1]): "Sideways",
    int(ranked_states[2]): "Bullish",
}
state_profiles["Assigned regime"] = [state_to_regime[int(s)] for s in state_profiles.index]
state_profiles = state_profiles.sort_values("Assigned regime")

display(state_profiles.round(6))
print("State-to-regime mapping:", state_to_regime)

## 6. Causal forward filtering of the holdout period

The holdout is processed sequentially. At each holdout candle, the posterior uses the previous filtered posterior, the learned transition matrix, and the current candle's emission likelihood. It does **not** use later holdout candles.

The code below evaluates the diagonal-Gaussian emission likelihood directly and applies the standard HMM forward recursion in log space for numerical stability.

In [ ]:
def _diag_variances_from_model(model):
    """Return Gaussian emission variances as (n_states, n_features)."""
    covars = np.asarray(model.covars_, dtype=float)
    if covars.ndim == 3:
        variances = np.diagonal(covars, axis1=1, axis2=2)
    elif covars.ndim == 2:
        variances = covars
    elif covars.ndim == 1:
        variances = covars[:, None]
    else:
        raise ValueError(f"Unexpected covariance array shape: {covars.shape}")
    if variances.shape != model.means_.shape:
        raise ValueError(
            f"Variance shape {variances.shape} does not match means shape {model.means_.shape}."
        )
    return np.maximum(variances, 1e-8)


def diagonal_gaussian_log_emissions(model, X):
    """Calculate log P(x_t | state) for each observation and hidden state."""
    X = np.asarray(X, dtype=float)
    means = np.asarray(model.means_, dtype=float)
    variances = _diag_variances_from_model(model)
    diff = X[:, None, :] - means[None, :, :]
    n_features = X.shape[1]
    log_det = np.log(variances).sum(axis=1)
    quadratic = (diff ** 2 / variances[None, :, :]).sum(axis=2)
    return -0.5 * (n_features * np.log(2.0 * np.pi) + log_det[None, :] + quadratic)


def forward_filter(model, X, initial_alpha=None):
    """Return filtered state probabilities, using current/past observations only.

    If initial_alpha is supplied, it is the filtered posterior at the previous
    observation. The transition matrix is applied before processing the first row
    in X. Otherwise, startprob_ initializes the first observation.
    """
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or len(X) == 0:
        raise ValueError("X must be a non-empty 2D feature matrix.")

    log_emissions = diagonal_gaussian_log_emissions(model, X)
    log_transition = np.log(np.clip(model.transmat_, 1e-300, 1.0))
    filtered = np.zeros((len(X), model.n_components), dtype=float)
    log_alpha = None if initial_alpha is None else np.log(np.clip(initial_alpha, 1e-300, 1.0))

    for t in range(len(X)):
        if log_alpha is None:
            log_alpha = np.log(np.clip(model.startprob_, 1e-300, 1.0)) + log_emissions[t]
        else:
            log_alpha = logsumexp(log_alpha[:, None] + log_transition, axis=0) + log_emissions[t]
        log_alpha = log_alpha - logsumexp(log_alpha)
        filtered[t] = np.exp(log_alpha)

    return filtered


# Run a forward-only filter through the training sequence to obtain the final
# filtered posterior at the training boundary. Then carry it into the holdout.
train_filtered_prob = forward_filter(best_model, X_train)
holdout_filtered_prob = forward_filter(
    best_model,
    X_holdout,
    initial_alpha=train_filtered_prob[-1],
)

if not np.allclose(holdout_filtered_prob.sum(axis=1), 1.0, atol=1e-6):
    raise AssertionError("Filtered state probabilities do not sum to one.")

print("Holdout forward-filtering completed.")

## 7. Create the out-of-sample HMM regime series

The regime name is assigned to the state with the highest filtered posterior probability at each holdout timestamp. Posterior confidence is retained as a diagnostic; it is not used to remove low-confidence observations in this baseline.

In [ ]:
holdout_state = holdout_filtered_prob.argmax(axis=1)
holdout_confidence = holdout_filtered_prob.max(axis=1)
hmm_oos = pd.DataFrame(index=holdout_features.index)
hmm_oos.index.name = "Date"
hmm_oos["HMM_State"] = holdout_state
hmm_oos["HMM_Regime"] = [state_to_regime[int(s)] for s in holdout_state]
hmm_oos["HMM_Confidence"] = holdout_confidence

for state_id in range(N_STATES):
    hmm_oos[f"P_State_{state_id}"] = holdout_filtered_prob[:, state_id]
for regime in REGIME_ORDER:
    state_ids = [s for s, label in state_to_regime.items() if label == regime]
    hmm_oos[f"P_{regime}"] = holdout_filtered_prob[:, state_ids].sum(axis=1)

regime_distribution = (
    hmm_oos["HMM_Regime"].value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .rename("Bars")
    .to_frame()
)
regime_distribution["Share %"] = 100 * regime_distribution["Bars"] / len(hmm_oos)
display(regime_distribution.round(2))
display(hmm_oos.head())

## 8. HMM regime diagnostics and comparison with the EMA/ADX baseline

The HMM and EMA/ADX classifier are different models, not competing ground-truth labels. The agreement table is descriptive only. Disagreement is expected because the baseline uses fixed indicator rules, while the HMM identifies latent statistical states from a multivariate feature distribution.

In [ ]:
# Learned transition matrix, reordered from numeric states into semantic labels.
transition = pd.DataFrame(
    best_model.transmat_,
    index=[state_to_regime[s] for s in range(N_STATES)],
    columns=[state_to_regime[s] for s in range(N_STATES)],
)
transition = transition.loc[REGIME_ORDER, REGIME_ORDER]
print("Training-sample model transition probabilities (mapped to regime names):")
display(transition.round(3))

baseline_oos = baseline_candles.reindex(hmm_oos.index)["Regime"]
regime_agreement = pd.crosstab(
    baseline_oos.rename("EMA_ADX_Regime"),
    hmm_oos["HMM_Regime"].rename("HMM_Regime"),
    normalize="index",
).reindex(index=["Bullish", "Bearish", "Sideways", "Unclassified"], columns=REGIME_ORDER)

agreement_rate = float((baseline_oos == hmm_oos["HMM_Regime"]).mean())
print(f"Raw label agreement with EMA/ADX baseline (descriptive, not accuracy): {agreement_rate:.1%}")
display(regime_agreement.fillna(0).round(3))

## 9. Visualize HMM regimes and posterior probabilities

These charts show only the chronological holdout segment. Regimes are inferred sequentially from filtered posterior probabilities, not from full-sample Viterbi decoding.

In [ ]:
# Price timeline coloured by the HMM's filtered regime.
plt.figure(figsize=(15, 6))
for regime in REGIME_ORDER:
    mask = hmm_oos["HMM_Regime"].eq(regime)
    plt.scatter(
        hmm_oos.index[mask],
        data.loc[hmm_oos.index[mask], "Close"],
        s=3,
        alpha=0.65,
        color=REGIME_COLORS[regime],
        label=regime,
    )
plt.title("XAUUSD H1 — HMM Regime Detection (Chronological Holdout)")
plt.xlabel("Timestamp")
plt.ylabel("Close price")
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(ASSET_DIR / "h1_hmm_oos_regime_timeline.png", dpi=160, bbox_inches="tight")
plt.show()

# Filtered regime posterior probabilities.
plt.figure(figsize=(15, 5))
for regime in REGIME_ORDER:
    plt.plot(
        hmm_oos.index,
        hmm_oos[f"P_{regime}"],
        label=f"P({regime})",
        linewidth=1.0,
        alpha=0.9,
    )
plt.title("XAUUSD H1 — HMM Filtered Regime Probabilities (Holdout)")
plt.xlabel("Timestamp")
plt.ylabel("Filtered probability")
plt.ylim(0, 1)
plt.grid(True, alpha=0.25)
plt.legend(ncol=3)
plt.tight_layout()
plt.savefig(ASSET_DIR / "h1_hmm_oos_regime_probabilities.png", dpi=160, bbox_inches="tight")
plt.show()

# Regime distribution in holdout.
ax = regime_distribution["Bars"].plot(
    kind="bar", figsize=(8, 4), color=[REGIME_COLORS[r] for r in REGIME_ORDER]
)
ax.set_title("XAUUSD H1 — HMM Holdout Regime Distribution")
ax.set_xlabel("HMM regime")
ax.set_ylabel("H1 bars")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(ASSET_DIR / "h1_hmm_oos_regime_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

## 10. Assign HMM regimes to N-Method trades

Each trade is assigned the HMM regime at its `signal_date`, when the setup was confirmed. For this out-of-sample analysis, only signals whose timestamps fall inside the HMM holdout period are retained. Ambiguous OCO cases are excluded, matching the earlier notebook's valid-trade convention.

In [ ]:
trades_with_hmm = trades_baseline.copy()
trades_with_hmm["signal_date"] = pd.to_datetime(trades_with_hmm["signal_date"])
trades_with_hmm = trades_with_hmm.drop(columns=[
    "HMM_State", "HMM_Regime", "HMM_Confidence"
], errors="ignore")

hmm_trade_labels = hmm_oos[["HMM_State", "HMM_Regime", "HMM_Confidence"]].copy()
trades_with_hmm = trades_with_hmm.merge(
    hmm_trade_labels,
    left_on="signal_date",
    right_index=True,
    how="left",
)

holdout_start = hmm_oos.index.min()
holdout_end = hmm_oos.index.max()
trades_oos = trades_with_hmm[
    trades_with_hmm["signal_date"].between(holdout_start, holdout_end)
].copy()
valid_oos = trades_oos[
    trades_oos["result"].isin(VALID_RESULTS) & trades_oos["HMM_Regime"].notna()
].copy()

ambiguous_oos = int((trades_oos["result"] == "AMBIGUOUS_BOTH_SIDES").sum())
print(f"Holdout trade records: {len(trades_oos):,}")
print(f"Valid holdout trades: {len(valid_oos):,}")
print(f"Ambiguous OCO records excluded: {ambiguous_oos:,}")
print(f"Signals outside HMM holdout omitted from performance analysis: {len(trades_with_hmm) - len(trades_oos):,}")
display(valid_oos[["setup", "signal_date", "HMM_Regime", "HMM_Confidence", "result", "R"]].head())

## 11. N-Method performance by HMM regime

Performance statistics are calculated only on trades signalled in the chronological holdout. These are historical conditional associations. The HMM was not optimized against the trade outcomes.

In [ ]:
def performance_stats(frame):
    frame = frame[frame["result"].isin(VALID_RESULTS)].copy()
    if frame.empty:
        return pd.Series({
            "Trades": 0,
            "Win rate %": np.nan,
            "Net R": 0.0,
            "Profit factor (R)": np.nan,
            "Avg R": np.nan,
        })
    gross_profit = frame.loc[frame["R"] > 0, "R"].sum()
    gross_loss = -frame.loc[frame["R"] < 0, "R"].sum()
    return pd.Series({
        "Trades": len(frame),
        "Win rate %": 100 * (frame["R"] > 0).mean(),
        "Net R": frame["R"].sum(),
        "Profit factor (R)": gross_profit / gross_loss if gross_loss > 0 else np.inf,
        "Avg R": frame["R"].mean(),
    })

strategy_order = ["bullish", "bearish", "ranging"]
conditional_rows = []
for strategy in strategy_order:
    for regime in REGIME_ORDER:
        subset = valid_oos[
            (valid_oos["setup"] == strategy) &
            (valid_oos["HMM_Regime"] == regime)
        ]
        conditional_rows.append({
            "Strategy": strategy.title() + " N",
            "HMM Regime": regime,
            **performance_stats(subset).to_dict(),
        })

hmm_regime_performance = pd.DataFrame(conditional_rows)
display(hmm_regime_performance.round(3))

pivot_hmm_net_r = hmm_regime_performance.pivot(
    index="Strategy", columns="HMM Regime", values="Net R"
).reindex(columns=REGIME_ORDER)
ax = pivot_hmm_net_r.plot(kind="bar", figsize=(10, 5))
ax.axhline(0, color="black", linewidth=0.9)
ax.set_title("XAUUSD H1 — N-Method Net R by HMM Regime (Holdout)")
ax.set_xlabel("")
ax.set_ylabel("Net R")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(ASSET_DIR / "h1_hmm_oos_net_r_by_strategy_regime.png", dpi=160, bbox_inches="tight")
plt.show()

## 12. Compare unfiltered, baseline-selected, and HMM-selected strategies

The selected configuration mirrors the current research hypothesis:

- Bullish N only when the detected regime is Bullish.
- Ranging N only when the detected regime is Sideways.
- Bearish N is excluded.

The comparison is restricted to the same HMM holdout signal period. The baseline-selected curve uses the existing EMA/ADX regime label from notebook 02; the HMM-selected curve uses the HMM's causal filtered regime.

In [ ]:
# Existing user-selected strategy mapping, applied separately to each classifier.
baseline_selected = valid_oos[
    ((valid_oos["setup"] == "bullish") & (valid_oos["Regime"] == "Bullish")) |
    ((valid_oos["setup"] == "ranging") & (valid_oos["Regime"] == "Sideways"))
].copy()

hmm_selected = valid_oos[
    ((valid_oos["setup"] == "bullish") & (valid_oos["HMM_Regime"] == "Bullish")) |
    ((valid_oos["setup"] == "ranging") & (valid_oos["HMM_Regime"] == "Sideways"))
].copy()

selection_sets = {
    "Unfiltered — all strategies": valid_oos,
    "EMA/ADX selected — Bullish N + Ranging N": baseline_selected,
    "HMM selected — Bullish N + Ranging N": hmm_selected,
}

selection_rows = []
for selection_name, subset in selection_sets.items():
    selection_rows.append({"Selection": selection_name, **performance_stats(subset).to_dict()})

selection_comparison = pd.DataFrame(selection_rows)
display(selection_comparison.round(3))

### Cumulative Net R over the same holdout period

Returns are grouped by trade exit timestamp before the cumulative sum. `OPEN_AT_END` trades are included at the end of the available sample as mark-to-market observations, not as realized closed trades. These are aggregate trade-level curves, not shared-capital portfolio simulations. Costs and portfolio exposure constraints are excluded.

In [ ]:
def cumulative_r_curve(frame, start_time):
    if frame.empty:
        return pd.Series(dtype=float)
    increments = frame.groupby("exit_date")["R"].sum().sort_index()
    increments.loc[pd.Timestamp(start_time)] = increments.get(pd.Timestamp(start_time), 0.0)
    increments = increments.groupby(level=0).sum().sort_index()
    return increments.cumsum()

start_time = holdout_start
curves = {
    name: cumulative_r_curve(frame, start_time)
    for name, frame in selection_sets.items()
}
curve_comparison = pd.concat(curves, axis=1).sort_index().ffill().fillna(0.0)

plt.figure(figsize=(15, 6))
for name in selection_sets:
    plt.plot(curve_comparison.index, curve_comparison[name], label=name, linewidth=1.8)
plt.axhline(0, color="black", linewidth=0.9, linestyle=":")
plt.title("XAUUSD H1 — HMM Holdout: Cumulative Net R Comparison")
plt.xlabel("Trade exit timestamp")
plt.ylabel("Cumulative Net R (risk units)")
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(ASSET_DIR / "h1_hmm_oos_cumulative_r_selection_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

display(curve_comparison.tail())

## 13. Interpretation checklist

Before drawing conclusions, inspect the following:

1. **State profiles:** Does the state mapped to Sideways actually have low average momentum and relatively low directional efficiency? Is one state primarily a volatility state?
2. **Regime persistence:** Does the transition matrix show stable, interpretable states, or frequent switching?
3. **Posterior confidence:** Are many holdout observations classified with weak posterior probabilities?
4. **Baseline agreement:** Where does HMM disagree with EMA/ADX, and do the differing labels correspond to visibly different market conditions?
5. **Conditional trade results:** Are the trade counts sufficiently large? Is any apparent edge driven by a small number of trades?
6. **Risk and costs:** Does performance remain positive after realistic spread, commission, swap, and slippage? Does it survive portfolio-level exposure constraints?

The state-to-regime naming convention is deliberately simple. If the state profiles do not resemble Bullish, Bearish, and Sideways conditions, do not force an economic interpretation simply because the model has three states.

## 14. Export HMM outputs

The exported regime labels cover only the chronological holdout period. The training sample is not assigned causal out-of-sample labels for this notebook's performance comparison.

In [ ]:
# Empirical holdout transition matrix for observed filtered labels.
holdout_transitions = pd.crosstab(
    hmm_oos["HMM_Regime"].shift(1).rename("From"),
    hmm_oos["HMM_Regime"].rename("To"),
).reindex(index=REGIME_ORDER, columns=REGIME_ORDER, fill_value=0)
holdout_transition_prob = holdout_transitions.div(
    holdout_transitions.sum(axis=1).replace(0, np.nan), axis=0
)

hmm_oos.to_csv(OUTPUT_DIR / "XAUUSD_H1_hmm_oos_regimes.csv")
state_profiles.to_csv(OUTPUT_DIR / "XAUUSD_H1_hmm_state_profiles.csv")
transition.to_csv(OUTPUT_DIR / "XAUUSD_H1_hmm_model_transition_matrix.csv")
holdout_transition_prob.to_csv(OUTPUT_DIR / "XAUUSD_H1_hmm_empirical_holdout_transition_matrix.csv")
regime_distribution.to_csv(OUTPUT_DIR / "XAUUSD_H1_hmm_regime_distribution.csv")
regime_agreement.to_csv(OUTPUT_DIR / "XAUUSD_H1_hmm_vs_ema_adx_agreement.csv")
hmm_regime_performance.to_csv(OUTPUT_DIR / "XAUUSD_H1_hmm_regime_performance.csv", index=False)
selection_comparison.to_csv(OUTPUT_DIR / "XAUUSD_H1_hmm_strategy_selection_comparison.csv", index=False)
trades_with_hmm.to_csv(OUTPUT_DIR / "XAUUSD_H1_N_method_trades_with_hmm_regime.csv", index=False)
curve_comparison.rename_axis("exit_date").to_csv(
    OUTPUT_DIR / "XAUUSD_H1_hmm_oos_cumulative_R_comparison.csv"
)
fit_diagnostics.to_csv(OUTPUT_DIR / "XAUUSD_H1_hmm_fit_diagnostics.csv", index=False)

print("HMM outputs saved to:")
for filename in [
    "XAUUSD_H1_hmm_oos_regimes.csv",
    "XAUUSD_H1_hmm_state_profiles.csv",
    "XAUUSD_H1_hmm_model_transition_matrix.csv",
    "XAUUSD_H1_hmm_empirical_holdout_transition_matrix.csv",
    "XAUUSD_H1_hmm_regime_distribution.csv",
    "XAUUSD_H1_hmm_vs_ema_adx_agreement.csv",
    "XAUUSD_H1_hmm_regime_performance.csv",
    "XAUUSD_H1_hmm_strategy_selection_comparison.csv",
    "XAUUSD_H1_N_method_trades_with_hmm_regime.csv",
    "XAUUSD_H1_hmm_oos_cumulative_R_comparison.csv",
    "XAUUSD_H1_hmm_fit_diagnostics.csv",
]:
    print(" -", OUTPUT_DIR / filename)

## 15. Limitations and next research steps

- **Single chronological split:** this is one fixed-parameter holdout experiment, not repeated walk-forward validation.
- **Latent states are not ground truth:** HMM identifies statistical states. Their Bullish/Bearish/Sideways names are assigned after fitting using training-period momentum.
- **Model risk:** Gaussian emissions and diagonal covariance are simplifying assumptions. Financial returns can be heavy-tailed and non-stationary.
- **No transaction costs:** spread, commission, swap, and slippage are excluded.
- **Trade-level aggregation:** cumulative Net R is not a shared-capital portfolio simulation and does not enforce overlapping-position limits, capital allocation, or margin constraints.
- **No parameter optimization:** three states and the chosen features are a baseline design, not an optimized configuration.
- **Regime mapping uncertainty:** the model's state labels may change with another training window or random initialization. State mapping must be re-estimated using training data only in any future refit.

### Research question

**Does HMM-based regime classification improve the N-Method's risk-adjusted performance on observations that were not used to fit or label the model?**

The next step should be a repeated walk-forward experiment with a pre-defined refit schedule, followed by transaction-cost-aware forward testing. These results remain **historical exploratory research, not a validated trading system**.